# Data Exploration — LaFrieda ERP Testbed

Explore the generated dummy data to verify realism and distributions.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from sqlalchemy import text
from database.connection import get_engine

engine = get_engine()
print('Connected to database')

In [ ]:
# Table row counts
tables = ['products', 'customers', 'suppliers', 'routes', 'lots', 'inventory',
          'invoices', 'invoice_line_items', 'purchase_orders', 'payments',
          'quality_records', 'campaigns', 'ar_aging', 'margin_summary']

with engine.connect() as conn:
    for t in tables:
        count = conn.execute(text(f'SELECT COUNT(*) FROM {t}')).scalar()
        print(f'{t:25s} {count:>10,}')

In [ ]:
# Product distribution by category
df = pd.read_sql('SELECT category, COUNT(*) as count FROM products GROUP BY category ORDER BY count DESC', engine)
df

In [ ]:
# Customer tier distribution
df = pd.read_sql('SELECT tier, COUNT(*) as count FROM customers GROUP BY tier', engine)
df

In [ ]:
# Monthly invoice volume
df = pd.read_sql("""
    SELECT strftime('%Y-%m', invoice_date) as month, 
           COUNT(*) as invoices, 
           ROUND(SUM(total_amount), 2) as revenue
    FROM invoices 
    GROUP BY month 
    ORDER BY month
""", engine)
df

In [ ]:
# Cari enrollment breakdown
df = pd.read_sql("""
    SELECT tier, 
           COUNT(*) as total,
           SUM(CASE WHEN cari_enrolled THEN 1 ELSE 0 END) as cari_enrolled,
           ROUND(100.0 * SUM(CASE WHEN cari_enrolled THEN 1 ELSE 0 END) / COUNT(*), 1) as pct
    FROM customers 
    GROUP BY tier
""", engine)
df

In [ ]:
# Lot status distribution
df = pd.read_sql('SELECT status, COUNT(*) as count FROM lots GROUP BY status', engine)
df